# 01 — Exact Solutions

Closed-form solutions for all four PDEs used as ground truth throughout the project.
No numerical solvers needed — every solution is a formula.

In [ ]:
import numpy as np
import torch
import math
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print('Exact Solutions Module Loaded.')

## 1. Heat Equation
u_t = alpha * u_xx  on periodic domain x in [0,1], t in [0,1]

Exact: u(x,t) = sum_k [a_k * sin(2pi*k*x) + b_k * cos(2pi*k*x)] * exp(-alpha*(2pi*k)^2 * t)

In [ ]:
def heat_exact_ic(x, k_max=16, seed=42):
    """Generate the Fourier-series initial condition for the heat equation.
    
    a_k, b_k ~ N(0, k^-2) so coefficients decay with wavenumber.
    This is the alpha_decay=2 case (smooth IC).
    """
    rng = torch.Generator()
    rng.manual_seed(seed)
    x = x.view(-1, 1) if x.dim() == 1 else x
    u = torch.zeros_like(x, dtype=torch.float32)
    for k in range(1, k_max + 1):
        a_k = torch.randn(1, generator=rng).item() * (k ** -2)
        b_k = torch.randn(1, generator=rng).item() * (k ** -2)
        u = u + a_k * torch.sin(2 * math.pi * k * x) + b_k * torch.cos(2 * math.pi * k * x)
    return u


def heat_exact_solution(x, t, alpha=0.01, k_max=16, seed=42):
    """Full exact heat equation solution at time t.
    
    Each mode decays at rate exp(-alpha * (2pi*k)^2 * t).
    Higher modes decay faster (spectral filtering).
    """
    rng = torch.Generator()
    rng.manual_seed(seed)
    x = x.view(-1, 1) if x.dim() == 1 else x
    u = torch.zeros_like(x, dtype=torch.float32)
    for k in range(1, k_max + 1):
        a_k = torch.randn(1, generator=rng).item() * (k ** -2)
        b_k = torch.randn(1, generator=rng).item() * (k ** -2)
        decay = math.exp(-alpha * (2 * math.pi * k) ** 2 * t)
        u = u + (a_k * torch.sin(2 * math.pi * k * x) + b_k * torch.cos(2 * math.pi * k * x)) * decay
    return u


# Verify: at t=0 IC and exact solution must match
x_test = torch.linspace(0, 1, 256)
u_ic   = heat_exact_ic(x_test)
u_t0   = heat_exact_solution(x_test, 0.0)
u_t1   = heat_exact_solution(x_test, 1.0)
print(f'IC vs t=0 max diff: {(u_ic - u_t0).abs().max().item():.2e}  (should be ~0)')

plt.figure(figsize=(10, 4))
plt.plot(x_test.numpy(), u_ic.squeeze().numpy(), label='t=0 (IC)')
plt.plot(x_test.numpy(), u_t1.squeeze().numpy(), label='t=1.0 (diffused)', linestyle='--')
plt.title('Heat Equation Exact Solution (alpha=0.01)')
plt.xlabel('x'); plt.ylabel('u(x,t)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Advection Equation
u_t + c * u_x = 0  on periodic domain x in [0,1]

Exact solution: rigid shift  u(x,t) = u0( (x - c*t) mod 1 )

The shape is preserved exactly — only translated at speed c.
This makes it an ideal test: if a PINN fails, it's not a capacity issue.

In [ ]:
def advection_exact_solution(x, t, c=1.0, k_max=16, seed=42):
    """Exact solution of the 1D advection equation u_t + c*u_x = 0.
    
    Periodic domain: x in [0,1].
    The solution is a rigid translation: u(x,t) = u0((x - c*t) mod 1).
    """
    x = x.view(-1, 1) if x.dim() == 1 else x
    x_shifted = (x - c * t) % 1.0
    return heat_exact_ic(x_shifted, k_max=k_max, seed=seed)


# Visualize at different speeds
x_test = torch.linspace(0, 1, 256)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, c in zip(axes, [1, 20, 50]):
    u0 = advection_exact_solution(x_test, 0.0, c=c)
    ut = advection_exact_solution(x_test, 1.0, c=c)
    ax.plot(x_test.numpy(), u0.squeeze().numpy(), label='t=0')
    ax.plot(x_test.numpy(), ut.squeeze().numpy(), label='t=1', linestyle='--')
    ax.set_title(f'Advection c={c}')
    ax.set_xlabel('x'); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Advection: Same Shape, Just Shifted (exact same IC)')
plt.tight_layout(); plt.show()
print('Note: c=50 wraps many times — PINN struggles here.')

## 3. Linear Water Waves
Velocity potential phi(x,z,t) satisfies:
  phi_xx + phi_zz = 0  (Laplace, interior)
  phi_z = 0  at z = -h  (seabed, no-flow)
  phi_tt + g*phi_z = 0  at z = 0  (free surface, linearized)

Exact solution:
  phi = (a*g/omega) * cosh(k*(z+h)) / cosh(k*h) * sin(k*x - omega*t)
  eta = a * cos(k*x - omega*t)

Dispersion relation: omega^2 = g * k * tanh(k*h)

Three regimes via kh:
  Shallow kh=0.3, Intermediate kh=1.0, Deep kh=3.0

In [ ]:
def water_wave_dispersion(k, h, g=9.81):
    """Angular frequency from linear dispersion relation: omega^2 = g*k*tanh(k*h)."""
    return math.sqrt(g * k * math.tanh(k * h))


def water_wave_exact(x, z, t, kh=1.0, a=0.01, g=9.81, L=1.0):
    """Exact travelling wave solution for linear water waves.
    
    Args:
        x: torch.Tensor -- horizontal coordinate
        z: torch.Tensor -- vertical coordinate (0 at surface, -h at bed)
        t: float -- time
        kh: float -- depth-to-wavelength ratio (controls regime)
        a: float -- wave amplitude
        g: float -- gravity
        L: float -- wavelength (domain length)

    Returns:
        phi: velocity potential
        eta: surface elevation (at z=0)
    """
    k = 2 * math.pi / L
    h = kh / k
    omega = water_wave_dispersion(k, h, g)

    cosh_num = torch.cosh(k * (z + h))
    cosh_den = math.cosh(k * h)
    phi = (a * g / omega) * (cosh_num / cosh_den) * torch.sin(k * x - omega * t)
    eta = a * torch.cos(k * x - omega * t)
    return phi, eta


# Print dispersion info
print('=== Dispersion Relation Summary ===')
for regime, kh in [('Shallow', 0.3), ('Intermediate', 1.0), ('Deep', 3.0)]:
    k = 2 * math.pi / 1.0
    h = kh / k
    omega = water_wave_dispersion(k, h)
    wave_speed = omega / k
    print(f'  {regime:15s}: kh={kh:.1f}, omega={omega:.4f}, c_phase={wave_speed:.4f}')

# Visualize surface elevation
x_test = torch.linspace(0, 1, 128)
z_surf = torch.zeros_like(x_test)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (regime, kh) in zip(axes, [('Shallow', 0.3), ('Intermediate', 1.0), ('Deep', 3.0)]):
    for t_val, style in [(0.0, '-'), (0.5, '--'), (1.0, ':')]:
        _, eta = water_wave_exact(x_test, z_surf, t_val, kh=kh)
        ax.plot(x_test.numpy(), eta.numpy(), style, label=f't={t_val}')
    ax.set_title(f'{regime} (kh={kh})')
    ax.set_xlabel('x'); ax.set_ylabel('eta')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Water Wave Surface Elevation (Linear Theory)')
plt.tight_layout(); plt.show()

## 4. Damped Harmonic Oscillator
z_tt + 2*zeta*omega0*z_t + omega0^2 * z = 0

Underdamped exact solution (zeta < 1):
  omega_d = omega0 * sqrt(1 - zeta^2)
  z(t) = exp(-zeta*omega0*t) * [C1*cos(omega_d*t) + C2*sin(omega_d*t)]

Default parameters: omega0=2, zeta=0.1
Train: t in [0, 5], Test: t in [5, 10] (and beyond)

In [ ]:
def oscillator_exact(t, z0=1.0, v0=0.0, omega0=2.0, zeta=0.1):
    """Exact underdamped oscillator solution.
    
    Args:
        t: torch.Tensor -- time points
        z0: float -- initial displacement z(0)
        v0: float -- initial velocity z_dot(0)
        omega0: float -- natural frequency
        zeta: float -- damping ratio (0 < zeta < 1 for underdamped)

    Returns:
        z: torch.Tensor -- position at each time point
    """
    assert zeta < 1.0, 'Only underdamped case supported'
    omega_d = omega0 * math.sqrt(1 - zeta ** 2)
    C1 = z0
    C2 = (v0 + zeta * omega0 * z0) / omega_d
    decay = torch.exp(-zeta * omega0 * t)
    return decay * (C1 * torch.cos(omega_d * t) + C2 * torch.sin(omega_d * t))


# Generate trajectories with various ICs
t_train = torch.linspace(0, 5, 100)
t_test  = torch.linspace(0, 10, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ics = [(1.0, 0.0), (0.5, 1.0), (-0.8, 0.5), (0.2, -1.2)]
for z0, v0 in ics:
    z = oscillator_exact(t_test, z0=z0, v0=v0)
    axes[0].plot(t_test.numpy(), z.numpy(), label=f'z0={z0}, v0={v0}')
axes[0].axvspan(5, 10, alpha=0.1, color='red', label='Test region')
axes[0].axvline(x=5, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Oscillator Trajectories (omega0=2, zeta=0.1)')
axes[0].set_xlabel('t'); axes[0].set_ylabel('z(t)')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Phase portrait
z_ph = oscillator_exact(t_test)
omega_d = 2.0 * math.sqrt(1 - 0.1**2)
z_dot_ph = (-0.1 * 2.0 * torch.exp(-0.1 * 2.0 * t_test) *
            (torch.cos(omega_d * t_test) + (0.1*2.0/omega_d)*torch.sin(omega_d * t_test)))
axes[1].plot(z_ph.numpy(), z_dot_ph.numpy())
axes[1].set_title('Phase Portrait (z0=1, v0=0)')
axes[1].set_xlabel('z'); axes[1].set_ylabel('dz/dt')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('All 4 exact solutions available and verified!')
print('Exported functions: heat_exact_ic, heat_exact_solution,')
print('                    advection_exact_solution, water_wave_exact,')
print('                    oscillator_exact, water_wave_dispersion')